# Inheritance
In the following example, two classes are introduced.
A car with technical parameters like consumption of fuel and amount of fuel.
A truck with the same technical parameters as the car. In Germany, trucks need to have a break after a certain amount of driving time. Therefore, the driving time since the last break is recorded in this class.

In [10]:
import numpy as np

eps = 1e-6

class CCar(object):
    
    def __init__(self):
        self.__ConsumptionPer100kmInLiter = 6.0
        self.__AmountOfFuelInLiter = 0.0
        self.__MaximumFuelInLiter = 60.0
        self.__DrivenKm = 0.0
        
    def EvaluateRange(self):
        return self.__AmountOfFuelInLiter * 100 / self.__ConsumptionPer100kmInLiter
    
    def Refuel(self, NewFuelInLiter):
        assert NewFuelInLiter >= 0.0, 'someone is stealing your fuel'
        self.__AmountOfFuelInLiter += NewFuelInLiter
        if self.__AmountOfFuelInLiter > self.__MaximumFuelInLiter:
            self.__AmountOfFuelInLiter = self.__MaximumFuelInLiter
            
    def Drive(self, NewKm):
        MaximumRange = self.EvaluateRange()
        if NewKm > MaximumRange:
            NewKm = MaximumRange
        self.__DrivenKm += NewKm
        self.__AmountOfFuelInLiter -= self.__ConsumptionPer100kmInLiter * NewKm / 100
            
class CTruck(object):
    
    def __init__(self):
        self.__ConsumptionPer100kmInLiter = 12.0
        self.__AmountOfFuelInLiter = 0.0
        self.__MaximumFuelInLiter = 200.0
        self.__DrivenKm = 0.0
        self.__DrivingTimeSinceLastBreakInHours = 0.0
        
    def EvaluateRange(self):
        return self.__AmountOfFuelInLiter * 100 / self.__ConsumptionPer100kmInLiter
    
    def Refuel(self, NewFuelInLiter):
        assert NewFuelInLiter >= 0.0, 'someone is stealing your fuel'
        self.__AmountOfFuelInLiter += NewFuelInLiter
        if self.__AmountOfFuelInLiter > self.__MaximumFuelInLiter:
            self.__AmountOfFuelInLiter = self.__MaximumFuelInLiter
            
    def Drive(self, NewKm, SpeedInKmPerHour):
        MaximumRange = self.EvaluateRange()
        if NewKm > MaximumRange:
            NewKm = MaximumRange
        self.__DrivenKm += NewKm
        self.__AmountOfFuelInLiter -= self.__ConsumptionPer100kmInLiter * NewKm / 100
        self.__DrivingTimeSinceLastBreakInHours += NewKm / SpeedInKmPerHour
        
    def MakeBreak(self):
        self.__DrivingTimeSinceLastBreakInHours = 0.0

ACar = CCar()
assert ACar.EvaluateRange() < eps, 'a new car has zero range (no fuel)'
ACar.Drive(NewKm = 100)
assert ACar.EvaluateRange() < eps, 'a new car has zero range and cannot drive (no fuel)'
ACar.Refuel(NewFuelInLiter=100)
assert ACar.EvaluateRange() > eps, 'a fueled car should have a range'
ATruck = CTruck()

Obviously, a lot of code is written twice for these two very similar classes.
On possibility to avoid these code duplications is inheritance, which is one of the basic principles for object orientation.
A class is defined, which defines the shared properties and procedures. In python, this class is called super-class.
From this super-class variations can be inherited.

In [11]:
class CCar(object):
    
    def __init__(self, ConsumptionPer100kmInLiter = 6.0, MaximumFuelInLiter = 60.0):
        self.__ConsumptionPer100kmInLiter = ConsumptionPer100kmInLiter
        self.__AmountOfFuelInLiter = 0.0
        self.__MaximumFuelInLiter = MaximumFuelInLiter
        self.__DrivenKm = 0.0
        
    def EvaluateRange(self):
        return self.__AmountOfFuelInLiter * 100 / self.__ConsumptionPer100kmInLiter
    
    def Refuel(self, NewFuelInLiter):
        assert NewFuelInLiter >= 0.0, 'someone is stealing your fuel'
        self.__AmountOfFuelInLiter += NewFuelInLiter
        if self.__AmountOfFuelInLiter > self.__MaximumFuelInLiter:
            self.__AmountOfFuelInLiter = self.__MaximumFuelInLiter
            
    def Drive(self, NewKm):
        MaximumRange = self.EvaluateRange()
        if NewKm > MaximumRange:
            NewKm = MaximumRange
        self.__DrivenKm += NewKm
        self.__AmountOfFuelInLiter -= self.__ConsumptionPer100kmInLiter * NewKm / 100
        return NewKm

class CTruck(CCar):
    
    def __init__(self):
        super().__init__(ConsumptionPer100kmInLiter = 12, MaximumFuelInLiter = 200)
        self.__DrivingTimeSinceLastBreakInHours = 0.0
            
    def Drive(self, NewKm, SpeedInKmPerHour):
        NewKm = super().Drive(NewKm)
        self.__DrivingTimeSinceLastBreakInHours += NewKm / SpeedInKmPerHour
        
    def MakeBreak(self):
        self.__DrivingTimeSinceLastBreakInHours = 0.0

    def GetDrivingTimeSinceLastBreakInHours(self):
        return self.__DrivingTimeSinceLastBreakInHours

ACar = CCar()
assert ACar.EvaluateRange() < eps, 'a new car has zero range (no fuel)'
ACar.Drive(NewKm = 100)
assert ACar.EvaluateRange() < eps, 'a new car has zero range and cannot drive (no fuel)'
ACar.Refuel(NewFuelInLiter=100)
assert ACar.EvaluateRange() > eps, 'a fueled car should have a range'
ATruck = CTruck()
ATruck.Refuel(100)
ATruck.Drive(100, 50)
assert ATruck.GetDrivingTimeSinceLastBreakInHours() > eps, 'error in evaluating the driving time sinve last break'
ATruck.MakeBreak()
assert ATruck.GetDrivingTimeSinceLastBreakInHours() < eps, 'error in resetting the driving time since last break'

## Programming exercise:

Write a third class COmnibus, which has all the properties of the class CTruck. The drivers of an omnibus needs to take care of its driving hours, too.
Additionally, the class COmnibus should have a property '__NumberOfPassengers', which should be accessed by a Getter- and a Setter-Procedure.

In [12]:
class COmnibus(CTruck):
    
    def __init__(self):
        super().__init__()
        self.__NumberOfPassengers = 0

    def GetNumberOfPassengers(self):
        return self.__NumberOfPassengers
    
    def SetNumberOfPassengers(self, NewNumberOfPassengers):
        assert NewNumberOfPassengers >= 0, 'negative number of passengers is not possible'
        self.__NumberOfPassengers = NewNumberOfPassengers

AOmnibus = COmnibus()
x = np.random.randint(100)
AOmnibus.SetNumberOfPassengers(x)
assert np.abs(AOmnibus.GetNumberOfPassengers() - x) < eps, 'error in getter or setter for NumberOfPassengers'
AOmnibus.Refuel(100)
AOmnibus.Drive(100, 33)
assert AOmnibus.GetDrivingTimeSinceLastBreakInHours() > eps, 'error in evaluating the driving time sinve last break'
AOmnibus.MakeBreak()
assert AOmnibus.GetDrivingTimeSinceLastBreakInHours() < eps, 'error in resetting the driving time since last break'